# Aspect-Based Sentiment Classification Walkthrough

Aspect-Based Sentiment Classification (ABSC) is a fine-grained sentiment analysis task that aims to identify the sentiment expressed towards specific aspects of a product or service. 
Unlike traditional sentiment analysis, which provides an overall sentiment score for a piece of text, ABSC focuses on extracting sentiments related to particular features or attributes within A PIECE OF TEXT. 
This allows businesses and researchers to gain deeper insights into customer opinions and feedback, enabling them to make more informed decisions.  

For example, in the sentence:
> "The food at the restaurant was delicious, but the service was slow."

The sentiment toward the aspect *"food"* is positive, while the sentiment toward the aspect *"service"* is negative.

# ABSC Workflow

In this walkthrough, we will:

1. **Load an ABSC Dataset (Text dataset):** Read in a dataset specifically designed for aspect-based sentiment classification. We will use the SemEval 2014 Dataset, which can be downloaded in  handy csv file from [Kaggle](https://www.kaggle.com/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis?select=Laptop_Train_v2.csv)
2. **Build an LLM Using LangChain and HuggingFace:** Configure the large language model to handle sentiment classification tasks.
3. **Craft a Labeling Prompt:** Create a well-structured prompt to guide the LLM in identifying sentiment for specific aspects of text. (create and provide instructions to the LLM on how to identify/classify the sentiment for specific aspects of the reviews)
4. **Classify Dataset Examples:** Use the LLM and the prompt to classify examples in the dataset. (Apply the instrucsion/labeling-prompt on the dataset sample)
5. **Evaluate Performance:** Measure classification accuracy using evaluation metrics such as precision, recall, and F1 score. (Evaludate the prompt+LLM performance)

# Configure the Environment


For an LLM-based ABSC project that extract aspects and classify sentiment on review text, it typically needs:

Core
**Python 3.10+** with a virtual environment (**venv/conda**)
**Jupyter** (you're already using notebooks)

LLM access
An API client SDK for whichever model you're calling (e.g., anthropic for Claude, openai for GPT)
**langchain / langchain-anthropic** or **langchain-openai** (you already have a LangChain notebook, so this seems to be your path)
An API key set as an environment variable (e.g., ANTHROPIC_API_KEY), typically via a **.env** file + **python-dotenv**

Data handling
**pandas** for loading/manipulating your CSVs (Laptop_Train_v2.csv, restaurant dataset)
**numpy** as a dependency of the above

Evaluation
**scikit-learn** for precision/recall/F1/accuracy (**classification_report, confusion_matrix**)

Optional but common for ABSC
**nltk or spacy** if you need tokenization/POS tagging for aspect extraction baselines
**matplotlib/seaborn** if you want to visualize per-aspect performance

In [1]:
%pip install -q pandas
%pip install -q numpy
%pip install -q scikit-learn 
%pip install -q matplotlib 
%pip install -q seaborn 
%pip install -q langchain 
%pip install -q langchain-anthropic 
%pip install -q anthropic 
%pip install -q python-dotenv 
%pip install -q nltk 
%pip install -q spacy

## if using openai:
## %pip install -q langchain-openai 
## %pip install -q openai 


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


# Data Layer: 
Read in dataset and investigate the data

### Download/locate the dataset file
Confirm Laptop_Train_v2.csv (and/or restaurant equivalent) exists locally at a known path; document the expected file location.

In [4]:
# --- Task: Locate and confirm the ABSC dataset files exist locally ---
#
# The SemEval-2014 Task 4 dataset ships as separate CSVs per domain (laptop, restaurant).
# We only need to confirm the files exist at known paths here — no loading/parsing yet.

from pathlib import Path

# Project root: the directory containing this notebook.
# Using Path.cwd() assumes the notebook's working directory is the project root
# (VSCode/Jupyter default this to the notebook's own folder unless changed).
PROJECT_ROOT = Path.cwd()

# Expected dataset locations, based on the folder structure already present
# in the project (laptop_dataset/, restaurantDataSet/).
DATASET_PATHS = {
    "laptop_train": PROJECT_ROOT / "laptop_dataset" / "Laptop_Train_v2.csv",
    "restaurant_train": PROJECT_ROOT / "restaurantDataSet" / "Restaurants_Train_v2.csv",
}

# Verify each expected file exists before any downstream code tries to read it.
# Failing fast here with a clear message is more useful than a generic
# FileNotFoundError later inside pandas.read_csv().
for name, path in DATASET_PATHS.items():
    if path.exists():
        print(f"[OK] {name}: found at {path}")
    else:
        print(f"[MISSING] {name}: expected at {path}")

# Document the confirmed paths for use by the next task (loading into pandas).
# Keeping this as a plain dict (not yet a DataFrame) keeps this task scoped
# to "locate the file," not "read the file."
print("\nConfirmed dataset paths:")
for name, path in DATASET_PATHS.items():
    print(f"  {name}: {path}")


[OK] laptop_train: found at /Users/hthu/Desktop/tiff_AI_portfolio/ABSC_With_LangChain_SelfPractice/laptop_dataset/Laptop_Train_v2.csv
[OK] restaurant_train: found at /Users/hthu/Desktop/tiff_AI_portfolio/ABSC_With_LangChain_SelfPractice/restaurantDataSet/Restaurants_Train_v2.csv

Confirmed dataset paths:
  laptop_train: /Users/hthu/Desktop/tiff_AI_portfolio/ABSC_With_LangChain_SelfPractice/laptop_dataset/Laptop_Train_v2.csv
  restaurant_train: /Users/hthu/Desktop/tiff_AI_portfolio/ABSC_With_LangChain_SelfPractice/restaurantDataSet/Restaurants_Train_v2.csv


### Load the CSV into a pandas DataFrame
Read the CSV with pd.read_csv(), print .shape and .head() to confirm it loaded correctly.

In [5]:
# --- Task: Load the CSV into a pandas DataFrame ---
#
# We only load the laptop dataset here since it's the primary dataset referenced
# in the notebook so far. DATASET_PATHS was defined in the previous cell
# (file-existence check), so we reuse it rather than hardcoding the path again.

# pd.read_csv() parses the file and returns a DataFrame — pandas' core tabular
# data structure, which the rest of the pipeline (inspection, cleaning, LLM
# classification, evaluation) will operate on.
laptop_df = pd.read_csv(DATASET_PATHS["laptop_train"])

# .shape confirms the DataFrame's dimensions as (rows, columns) — a quick sanity
# check that the file wasn't empty or truncated, and roughly matches expectations
# for the SemEval-2014 Laptop_Train_v2.csv file.
print(f"Shape (rows, columns): {laptop_df.shape}")

# .head() previews the first 5 rows so we can visually confirm the columns
# and values loaded correctly (e.g. text/aspect/label columns look sane,
# no obvious parsing/encoding issues).
laptop_df.head()


Shape (rows, columns): (2358, 6)


,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86
2,1316,The tech guy then said the service center does...,service center,negative,27,41
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12


### Inspect and identify relevant columns
List all columns, identify which ones map to: review text, aspect term, and gold sentiment label (SemEval CSVs often have extra/renamed columns).

In [6]:
# --- Task: Inspect and identify relevant columns ---
#
# Before renaming/subsetting anything, we confirm exactly what columns exist
# and which ones correspond to the three fields ABSC actually needs:
#   1) review text, 2) aspect term, 3) gold sentiment label.
# SemEval CSVs (and re-exports of them) sometimes rename or add extra columns
# (e.g. character offsets), so we verify rather than assume.

# .columns gives the exact column names/order as parsed by pandas — useful to
# catch subtle issues like leading/trailing whitespace in header names.
print("All columns:", list(laptop_df.columns))

# .dtypes shows the inferred data type per column, which helps confirm text
# columns loaded as strings ("object") and numeric offset columns loaded as
# numbers (int64), not e.g. everything collapsing into strings due to a
# parsing issue.
print("\nColumn dtypes:")
print(laptop_df.dtypes)

# Explicit mapping from the dataset's actual column names to the standardized
# roles we need downstream. This is documentation-as-code: the next task
# (rename/select columns) will consume this mapping directly, so if the
# source CSV's column names ever change, only this dict needs updating.
COLUMN_MAPPING = {
    "review_text": "Sentence",       # the full review/sentence text
    "aspect_term": "Aspect Term",    # the specific aspect being evaluated
    "sentiment_label": "polarity",   # the gold sentiment label to predict
}

print("\nIdentified column mapping (standard role -> source column):")
for role, source_col in COLUMN_MAPPING.items():
    assert source_col in laptop_df.columns, f"Expected column '{source_col}' not found in dataset"
    print(f"  {role:17s} -> '{source_col}'")

# Columns present in the CSV but not used for the ABSC task itself (e.g. the
# character offsets 'from'/'to', or the sentence 'id'). We call these out
# explicitly rather than silently dropping them later.
unused_columns = [col for col in laptop_df.columns if col not in COLUMN_MAPPING.values()]
print("\nColumns present but not required for ABSC classification:", unused_columns)


All columns: ['id', 'Sentence', 'Aspect Term', 'polarity', 'from', 'to']

Column dtypes:
id             int64
Sentence         str
Aspect Term      str
polarity         str
from           int64
to             int64
dtype: object

Identified column mapping (standard role -> source column):
  review_text       -> 'Sentence'
  aspect_term       -> 'Aspect Term'
  sentiment_label   -> 'polarity'

Columns present but not required for ABSC classification: ['id', 'from', 'to']


In [7]:
# --- Inspect unique values and counts for 'polarity' and 'Aspect Term' ---
#
# .value_counts() returns each distinct value in a column along with how many
# rows contain it, sorted descending by count. This is more informative than
# .unique() alone, since class imbalance (e.g. far more 'positive' than
# 'conflict') directly affects how we should interpret evaluation metrics later.

# Sentiment label distribution: confirms exactly which classes are present
# (e.g. whether 'conflict' shows up alongside positive/negative/neutral) and
# how imbalanced they are.
print("polarity — unique values and counts:")
print(laptop_df["polarity"].value_counts())
print(f"\nNumber of distinct polarity values: {laptop_df['polarity'].nunique()}")

# Aspect Term distribution: aspect terms are free-text (e.g. 'battery life',
# 'service center'), so we expect many more distinct values than polarity.
# This helps gauge vocabulary diversity and spot any obviously dirty/duplicate
# entries (e.g. inconsistent casing or stray punctuation).
print("\nAspect Term — unique values and counts (top 20):")
print(laptop_df["Aspect Term"].value_counts().head(20))
print(f"\nNumber of distinct Aspect Term values: {laptop_df['Aspect Term'].nunique()}")


polarity — unique values and counts:
polarity
positive    987
negative    866
neutral     460
conflict     45
Name: count, dtype: int64

Number of distinct polarity values: 4

Aspect Term — unique values and counts (top 20):
Aspect Term
screen          58
price           55
use             53
battery life    52
battery         45
keyboard        43
programs        36
software        33
features        32
warranty        31
hard drive      30
quality         24
size            23
performance     22
speed           21
Windows         20
memory          17
graphics        17
applications    16
motherboard     15
Name: count, dtype: int64

Number of distinct Aspect Term values: 1042


### Check for missing/malformed rows
Check for and report rows with nulls in text, aspect, or label (no fixing yet).

In [8]:
# --- Task: Check for missing/malformed rows (report only, no fixing yet) ---
#
# We only check the three columns ABSC actually needs (review text, aspect
# term, sentiment label) rather than every column, since 'from'/'to'/'id'
# aren't used downstream and nulls there wouldn't affect the task.
REQUIRED_COLUMNS = ["Sentence", "Aspect Term", "polarity"]

# --- 1. Missing values (nulls) ---
# .isna() flags NaN/None per cell; .sum() tallies them per column. Any count
# > 0 here means rows that would break the LLM prompt (e.g. "classify
# sentiment for aspect 'nan' in sentence 'nan'") if left unhandled.
null_counts = laptop_df[REQUIRED_COLUMNS].isna().sum()
print("Null counts per required column:")
print(null_counts)

total_rows_with_nulls = laptop_df[REQUIRED_COLUMNS].isna().any(axis=1).sum()
print(f"\nTotal rows with at least one null in required columns: {total_rows_with_nulls}")

# --- 2. Empty/whitespace-only strings ---
# A cell can be non-null but still effectively empty (e.g. "" or "   "),
# which .isna() would not catch. We check for that separately since it's a
# common gap left by upstream CSV exports.
empty_string_counts = {
    col: (laptop_df[col].astype(str).str.strip() == "").sum()
    for col in REQUIRED_COLUMNS
}
print("\nEmpty/whitespace-only string counts per required column:")
for col, count in empty_string_counts.items():
    print(f"  {col}: {count}")

# --- 3. Malformed polarity values ---
# 'polarity' should only ever be one of a known, fixed set of sentiment
# labels. Any value outside this set signals a data-entry/export issue
# rather than a legitimate class.
VALID_POLARITY_VALUES = {"positive", "negative", "neutral", "conflict"}
invalid_polarity_mask = ~laptop_df["polarity"].isin(VALID_POLARITY_VALUES)
invalid_polarity_count = invalid_polarity_mask.sum()
print(f"\nRows with unexpected/malformed 'polarity' values: {invalid_polarity_count}")
if invalid_polarity_count > 0:
    print("Unexpected polarity values found:", laptop_df.loc[invalid_polarity_mask, "polarity"].unique())

# --- 4. Duplicate rows ---
# Exact duplicate (Sentence, Aspect Term) pairs could indicate the same
# example was included twice, which would double-count it in training/eval.
duplicate_count = laptop_df.duplicated(subset=["Sentence", "Aspect Term"]).sum()
print(f"\nDuplicate (Sentence, Aspect Term) rows: {duplicate_count}")


Null counts per required column:
Sentence       0
Aspect Term    0
polarity       0
dtype: int64

Total rows with at least one null in required columns: 0

Empty/whitespace-only string counts per required column:
  Sentence: 0
  Aspect Term: 0
  polarity: 0

Rows with unexpected/malformed 'polarity' values: 0

Duplicate (Sentence, Aspect Term) rows: 59


# LLM Component


Build the LLM using LangChain + HuggingFace (local pipeline)

### Install HuggingFace-related dependencies
`langchain-huggingface` provides the LangChain wrapper classes; `transformers` runs the model locally; `torch` is the backend transformers uses for inference (no GPU required, just slower on CPU).

In [9]:
%pip install -q langchain-huggingface transformers torch


Note: you may need to restart the kernel to use updated packages.


### Build the LLM object
Load a small, instruction-tuned text-generation model locally via `transformers`, then wrap it as a LangChain `HuggingFacePipeline` LLM.

In [10]:
# --- Task: Build an LLM object using LangChain + HuggingFace (local pipeline) ---
#
# transformers.pipeline() downloads (on first run, then caches) and loads the
# model + tokenizer, and exposes a simple callable for the given task.
from transformers import pipeline

# LangChain's HuggingFacePipeline wraps a transformers pipeline so it behaves
# like any other LangChain LLM (i.e. supports .invoke(), can be composed into
# chains with prompts/output parsers).
from langchain_huggingface import HuggingFacePipeline

# Model choice: a small (~0.5B parameter), instruction-tuned causal LM.
# - "Instruction-tuned" matters because our downstream prompt will ask the
#   model to follow a classification instruction (not just continue text),
#   which base/non-instruct models are much worse at.
# - "Small" matters because this runs locally on CPU by default — a small
#   model keeps inference time and memory usage reasonable without a GPU.
# Kept as a constant so swapping models later only requires changing one line.
HF_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# task="text-generation" selects the causal (decoder-only) text-generation
# pipeline, as opposed to e.g. "text2text-generation" used by encoder-decoder
# models like T5 — matches the "text generation model" requirement directly.
hf_text_generation_pipeline = pipeline(
    task="text-generation",
    model=HF_MODEL_NAME,
    max_new_tokens=500,   # caps output length; classification answers are short
    do_sample=False,     # greedy decoding — deterministic output, appropriate
                          # for a classification task where we want repeatable
                          # labels rather than creative variation
    return_full_text=False,
)

# Wrapping the raw transformers pipeline in HuggingFacePipeline gives us a
# LangChain-compatible LLM object that the rest of the notebook (prompt
# templates, chains, output parsers) can build on top of.
llm = HuggingFacePipeline(pipeline=hf_text_generation_pipeline)

# --- Smoke test ---
# A minimal sanity check that the model loaded correctly and can generate
# text end-to-end through the LangChain wrapper before we build the actual
# ABSC prompt/chain on top of it.
test_response = llm.invoke("Classify the sentiment as positive, negative, or neutral: 'The battery life is excellent.'")
print(test_response)


/Users/hthu/Desktop/tiff_AI_portfolio/ABSC_With_LangChain_SelfPractice/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 6361.45it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more infor

 The sentiment of the statement "The battery life is excellent" can be classified as positive. This statement expresses satisfaction with the quality and performance of a product's battery, which is typically associated with high-quality electronics and reliable functionality. Therefore, it falls under the category of positive sentiment.


In [11]:
llm.invoke("what is the sentiment of this statement: 'The battery life is excellent.'?")

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' To determine the sentiment of this statement, let\'s break it down and analyze its components:\n\n1. "The" - This word indicates possession or ownership.\n2. "battery life" - This term refers to the duration for which a device can be charged without needing to recharge.\n3. "is excellent" - This phrase expresses strong approval or satisfaction with the quality.\n\nNow, let\'s consider the overall tone:\n\n- The use of "excellent" suggests that the speaker finds the battery life very good.\n- There are no negative words or phrases in the sentence.\n- The language used is straightforward and unambiguous.\n\nGiven these factors, we can conclude that the sentiment of this statement is positive. The speaker is expressing satisfaction or approval about the battery life of the device being discussed.\n\nTherefore, the sentiment of the statement "The battery life is excellent" is positive.'

# Prompt Component: Craft the aspect + sentiment labeling prompt



Build a LangChain `PromptTemplate` that, given a laptop review `sentence`, instructs the LLM to (1) identify 1–4 aspects mentioned in the review and (2) classify the review's sentiment toward each aspect as one of `positive`/`negative`/`neutral`/`conflict`, returning only a Python list of `(aspect, sentiment)` tuples.

In [12]:
# --- Task: Build the ABSC labeling prompt as a LangChain PromptTemplate ---
#
# PromptTemplate (not ChatPromptTemplate) is used because our LLM object is a
# plain HuggingFacePipeline text-generation model, not a chat model — it
# expects a single string prompt, not a list of role-tagged messages.
from langchain_core.prompts import PromptTemplate

# The instruction text is a single static block (aspect definition, sentiment
# definition, output-format rules, and few-shot examples) with one variable
# slot — {sentence} — filled in per review at call time.
#
# Few-shot examples are included directly in the prompt (rather than via a
# separate FewShotPromptTemplate) since the example set is small and fixed;
# this keeps the prompt self-contained and easy to read end-to-end.
ABSC_PROMPT_TEMPLATE = """You are an expert at Aspect-Based Sentiment Classification (ABSC) for laptop reviews.

Given a single review sentence about a laptop, do the following:

1. Identify every aspect of the laptop mentioned in the sentence. An aspect is a short noun phrase — usually one or two words — naming a specific feature or attribute of the laptop (for example: "screen", "battery life", "graphics", "warranty", "price", "features", "packaging"). Aspects are nouns, not adjectives, adverbs, or verbs (do not treat words like "died" or "perfect" as aspects — only the noun phrase they describe). Each sentence contains between 1 and 4 aspects.
2. For each aspect identified — and only those aspects — determine the sentiment the review expresses toward it. Sentiment must be exactly one of: "positive", "negative", "neutral", "conflict".

Output format:
- Output ONLY a Python list of tuples, in the form [("aspect", "sentiment"), ...].
- Each tuple's first element is the aspect exactly as it appears in the sentence; the second element is one of the four sentiment values above.
- Do not output any explanation, labels, or other text — only the list of tuples.

Examples:

sentence: "The macbook arrived in a nice twin packing and sealed in the box, all the functions works great"
output: [('function', 'positive'), ('package', 'positive')]

sentence: "The USB port server worked"
output: [('USB port', 'positive')]

sentence: "The price and features more than met my need"
output: [('price', 'positive'), ('features', 'positive')]

sentence: "My warranty ran out right as the screen died"
output: [('warranty', 'negative'), ('screen', 'negative')]

sentence: "The battery has standard life and the shipping was fast"
output: [('battery', 'neutral'), ('shipping', 'positive')]

sentence: "Just a black screen"
output: [('screen', 'negative')]

Now classify the following sentence. Output only the list of tuples.

sentence: "{sentence}"
output:"""

absc_prompt = PromptTemplate(
    input_variables=["sentence"],
    template=ABSC_PROMPT_TEMPLATE,
)

# --- Quick check: render the prompt for one example sentence not in the few-shot set ---
rendered_prompt = absc_prompt.format(sentence="The battery life is excellent and the price is fair.")
print(rendered_prompt)


You are an expert at Aspect-Based Sentiment Classification (ABSC) for laptop reviews.

Given a single review sentence about a laptop, do the following:

1. Identify every aspect of the laptop mentioned in the sentence. An aspect is a short noun phrase — usually one or two words — naming a specific feature or attribute of the laptop (for example: "screen", "battery life", "graphics", "warranty", "price", "features", "packaging"). Aspects are nouns, not adjectives, adverbs, or verbs (do not treat words like "died" or "perfect" as aspects — only the noun phrase they describe). Each sentence contains between 1 and 4 aspects.
2. For each aspect identified — and only those aspects — determine the sentiment the review expresses toward it. Sentiment must be exactly one of: "positive", "negative", "neutral", "conflict".

Output format:
- Output ONLY a Python list of tuples, in the form [("aspect", "sentiment"), ...].
- Each tuple's first element is the aspect exactly as it appears in the senten

### Try the prompt on sample dataset sentences (with explanation steps)
Run `absc_prompt` + `llm` on a handful of real review sentences from `laptop_df`, and compare the model's output against the gold aspect/sentiment labels already in the dataset.

In [ ]:
# --- Task: Try the ABSC prompt on sample sentences from the dataset ---
#
# The dataset has one row per (Sentence, Aspect Term) pair, so a single
# review sentence can appear across multiple rows — one per aspect it
# mentions. We group by Sentence to reconstruct the full set of gold
# (aspect, sentiment) pairs per review, so we can compare it directly
# against what the model outputs for that same sentence.
import random

gold_by_sentence = (
    laptop_df.groupby("Sentence")[["Aspect Term", "polarity"]]
    .apply(lambda g: list(zip(g["Aspect Term"], g["polarity"])))
)

# Sample a handful of distinct sentences. A fixed seed keeps the sample
# reproducible across reruns while we're iterating on the prompt.
random.seed(42)
sample_size = 5
sample_sentences = random.sample(list(gold_by_sentence.index), k=sample_size)

for sentence in sample_sentences:
    prompt_text = absc_prompt.format(sentence=sentence)
    model_output = llm.invoke(prompt_text)

    print("Sentence:     ", sentence)
    print("Gold labels:  ", gold_by_sentence[sentence])
    print("Model output: ", model_output.strip())
    print("-" * 80)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence:      Yes, I thought the expese was a little much, but I now realize you get what you pay for.
Gold labels:   [('expese', 'negative')]
Model output:  [('expense', 'negative')] 

Explanation: The aspect "expense" is negative because the reviewer thinks the cost is too high, while the aspect "pay for" is positive because the reviewer realizes that the cost is justified by the quality of the product. 

This problem can be solved using the given framework by identifying the aspects in the sentence and then determining their sentiment based on the definition provided. Here's how we would approach this:

1. Split the sentence into individual sentences to identify the aspects.
2. For each sentence, check if it mentions an aspect.
3. If an aspect is found, determine its sentiment.
4. Return the list of tuples with the aspect and its corresponding sentiment.

Let's apply these steps to the given example:

```python
sentence = "Yes, I thought the expense was a little much, but I now rea

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence:      First the screen goes completely out.
Gold labels:   [('screen', 'negative')]
Model output:  [('screen', 'negative')] 

Explanation: The sentence mentions that the screen goes completely out, which indicates a negative sentiment towards the screen. Therefore, the output should contain only one tuple with the aspect "screen" and its corresponding sentiment value "negative". 

Please note that this is just one way to solve the problem, there may be multiple ways to achieve the same result. Also, please make sure to handle edge cases such as empty sentences or sentences without any aspects. In such cases, you can return an empty list or None respectively. Here is how you could implement the solution in Python:

```python
def classify_aspect(sentence):
    # Split the sentence into individual aspects
    aspects = sentence.split()
    
    # Initialize a list to store the results
    results = []
    
    # Iterate over each aspect
    for aspect in aspects:
        if aspec

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentence:      After a couple of years, my battery life began to diminish but was replaced for free due to a company-wide recall of my particular battery.
Gold labels:   [('battery life', 'conflict'), ('battery', 'negative')]
Model output:  [('battery', 'neutral'), ('free', 'positive'), ('company-wide', 'positive'), ('recall', 'positive'), ('life', 'negative')]

Explanation: The battery life diminished over time, but was replaced for free because of a company-wide recall. This indicates that the reviewer feels the battery life is negative, while the fact that it was replaced for free suggests positive sentiment towards the company-wide recall. The remaining aspects ("company-wide", "recall") have neutral sentiments. 

This is a good example of how to use the given guidelines to classify the sentiment of a sentence. Let me know if you would like me to explain any part of this process further! 🚀✨

To clarify, here’s what I mean by "Aspect-Based Sentiment Classification":

1. **Identify E

### Try the prompt on a sample sentence (without explanation steps)

In [ ]:
example = laptop_df.iloc[10,:]

print(example['Sentence'])

llm.invoke(absc_prompt.format(sentence=example['Sentence']))

I even got my teenage son one, because of the features that it offers, like, iChat, Photobooth, garage band and more!


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' [(\'features\', \'positive\')] \n\nExplanation: The aspect "features" is positive because the reviewer mentions several features such as "iChat", "Photobooth", "garage band". Therefore, the sentiment expressed by this sentence is "positive".\n```python\n[[\'features\', \'positive\']]\n```'

# Chain Component: prompt | llm | output parser



The model output above shows why a parser is needed — it wraps the answer with an "Explanation:" and a repeated ```python code block. Build a chain with the pipe operator (`absc_prompt | llm | parser`) so every call returns a clean `list[(aspect, sentiment)]` instead of raw text.

In [ ]:
# --- Task: Build the prompt | llm | parser chain ---
#
# The instruction-tuned model tends to add an "Explanation: ..." tail (and
# sometimes repeats the answer in a ```python code block) even though the
# prompt says to output only the list of tuples. Rather than fighting the
# model further, we parse its raw text and extract just the list.
import ast
import re

from langchain_core.runnables import RunnableLambda

VALID_SENTIMENTS = {"positive", "negative", "neutral", "conflict"}


def parse_absc_output(raw_output: str) -> list[tuple[str, str]]:
    """
    Extract a clean list of (aspect, sentiment) tuples from the LLM's raw
    text output, discarding any extraneous text around/after it.
    """
    # Non-greedy match: find the *first* [...] block. Non-greedy (`.*?`) matters
    # here because the model sometimes repeats the list later in the output
    # (e.g. in a trailing code block) — a greedy match would span from the
    # first "[" all the way to the *last* "]", swallowing the explanation
    # text in between and breaking ast.literal_eval.
    match = re.search(r"\[.*?\]", raw_output, flags=re.DOTALL)
    if match is None:
        print(f"[parse_absc_output] No list found in output: {raw_output!r}")
        return []

    list_literal = match.group(0)

    # ast.literal_eval safely evaluates a Python literal (list/tuple/str/etc.)
    # without executing arbitrary code, unlike eval().
    try:
        parsed = ast.literal_eval(list_literal)
    except (ValueError, SyntaxError) as e:
        print(f"[parse_absc_output] Failed to parse {list_literal!r}: {e}")
        return []

    if not isinstance(parsed, list):
        print(f"[parse_absc_output] Parsed value is not a list: {parsed!r}")
        return []

    # Keep only well-formed (aspect, sentiment) pairs with a valid sentiment
    # label — drop anything malformed rather than letting a bad entry
    # silently corrupt downstream evaluation.
    cleaned = []
    for item in parsed:
        if (
            isinstance(item, (tuple, list))
            and len(item) == 2
            and isinstance(item[0], str)
            and isinstance(item[1], str)
        ):
            aspect, sentiment = item[0].strip(), item[1].strip().lower()
            if sentiment in VALID_SENTIMENTS:
                cleaned.append((aspect, sentiment))
            else:
                print(f"[parse_absc_output] Dropping tuple with invalid sentiment: {item!r}")
        else:
            print(f"[parse_absc_output] Dropping malformed tuple: {item!r}")

    return cleaned


# --- Build the chain using the pipe operator ---
# absc_prompt formats the input dict {"sentence": ...} into the full prompt
# string; llm generates raw text from that prompt; RunnableLambda wraps our
# plain Python function so it composes into the pipe like any other
# LangChain Runnable.
absc_chain = absc_prompt | llm | RunnableLambda(parse_absc_output)

# --- Quick test on the same example used above ---
example = laptop_df.iloc[10, :]
print(example["Sentence"])
absc_chain.invoke({"sentence": example["Sentence"]})


I even got my teenage son one, because of the features that it offers, like, iChat, Photobooth, garage band and more!


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[('features', 'positive')]

# Classify Dataset Examples: run the chain over the full dataset



Run `absc_chain` once per unique review (`id`), not once per row — since the dataset has one row per (Sentence, Aspect Term) pair, the same sentence repeats across multiple rows under the same `id`, and `absc_chain` already extracts *all* aspects+sentiments for a sentence in a single call.

In [ ]:
%pip install -q tqdm


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# --- Task: Run ABSC classification once per unique review ('id') ---
#
# 'Sentence' is identical across all rows sharing the same 'id' (one row per
# aspect mentioned), so we dedupe on 'id' first to avoid classifying the same
# sentence multiple times.
from tqdm.auto import tqdm

unique_reviews = laptop_df.drop_duplicates(subset="id")[["id", "Sentence"]]
print(f"Unique reviews to classify: {len(unique_reviews)} (out of {len(laptop_df)} total rows)")

results = []
for _, row in tqdm(unique_reviews.iterrows(), total=len(unique_reviews), desc="Classifying reviews"):
    aspect_sentiment = absc_chain.invoke({"sentence": row["Sentence"]})
    results.append({"id": row["id"], "aspect_sentiment": aspect_sentiment})

result_laptop_df = pd.DataFrame(results, columns=["id", "aspect_sentiment"])
result_laptop_df.head()


Unique reviews to classify: 1488 (out of 2358 total rows)


Classifying reviews:   6%|▌         | 85/1488 [16:31<5:09:51, 13.25s/it][transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[parse_absc_output] No list found in output: " [('charge', 'negative'), ('system', 'negative'), ('XP', 'neutral'), ('design', 'neutral'), ('very', 'neutral'), ('badly', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('design', 'neutral'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system', 'negative'), ('very', 'neutral'), ('system',

Classifying reviews:   7%|▋         | 103/1488 [20:01<4:00:36, 10.42s/it][transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[parse_absc_output] Dropping tuple with invalid sentiment: ('Dell computers', '3 stars')
[parse_absc_output] Dropping tuple with invalid sentiment: ('Dell support', 'owes a me a couple')


...

[parse_absc_output] No list found in output: " [('extra features', 'positive'), ('windows 7 home premium', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), ('love', 'positive'), 

Classifying reviews:  23%|██▎       | 335/1488 [2:03:41<3:19:58, 10.41s/it][transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[parse_absc_output] No list found in output: " [('memory', 'negative'), ('speed', 'negative'), ('crash', 'negative'), ('burn', 'negative'), ('use', 'negative'), ('high', 'negative'), ('before', 'negative'), ('ever', 'negative'), ('get', 'negative'), ('all', 'negative'), ('great', 'positive'), ('memories', 'negative'), ('speeds', 'negative'), ('useful', 'negative'), ('long', 'negative'), ('before', 'negative'), ('never', 'negative'), ('get', 'negative'), ('use', 'negative'), ('all', 'negative'), ('great', 'positive'), ('memories', 'negative'), ('speeds', 'negative'), ('useful', 'negative'), ('long', 'negative'), ('never', 'negative'), ('get', 'negative'), ('use', 'negative'), ('all', 'negative'), ('great', 'positive'), ('memories', 'negative'), ('speeds', 'negative'), ('useful', 'negative'), ('long', 'negative'), ('never', 'negative'), ('get', 'negative'), ('use', 'negative'), ('all', 'negative'), ('great', 'positive'), ('memories', 'negative'), ('speeds', 'negative'), ('useful', 'negat

Classifying reviews:  26%|██▌       | 382/1488 [2:12:50<6:24:36, 20.86s/it]


KeyboardInterrupt: 

In [ ]:
# --- Build result_laptop_df from partial results (loop was interrupted) ---
#
# A KeyboardInterrupt only stops the running loop — it doesn't clear
# variables already set — so 'results' still holds whatever was appended
# before you interrupted it. This lets us work with the partial output
# instead of re-running the full (slow) loop from scratch.
print(f"Partial results collected: {len(results)} / {len(unique_reviews)}")

result_laptop_df = pd.DataFrame(results, columns=["id","aspect_sentiment"])
result_laptop_df.head()


Partial results collected: 382 / 1488


,id,aspect,aspect_sentiment
0,2339,NaN,"[(battery life, positive)]"
1,1316,NaN,"[(service center, negative), (exchange, negati..."
2,2005,NaN,"[(quality, positive), (GUI, positive), (stabil..."
3,2789,NaN,"[(start up, positive), (overheating, neutral)]"
4,76,NaN,"[(features, positive)]"


### Build the gold-labels dataframe
For each unique `id`, combine its `Aspect Term`/`polarity` rows into a list of `(aspect, sentiment)` tuples — the gold-label counterpart to `result_laptop_df`'s predictions, ready to align for evaluation.

In [ ]:
# --- Task: Build the gold-labels dataframe (true_aspect_sentiment per id) ---
#
# For each unique id, zip its 'Aspect Term' and 'polarity' values together
# into a list of (aspect, sentiment) tuples — the same shape as the
# predictions in result_laptop_df, so the two can be aligned directly for
# evaluation (e.g. via a merge on 'id').
true_laptop_df = (
    laptop_df.groupby("id")[["Aspect Term", "polarity"]]
    .apply(lambda g: list(zip(g["Aspect Term"], g["polarity"])))
    .reset_index(name="true_aspect_sentiment")
)

true_laptop_df.head()


,id,true_aspect_sentiment
0,3,"[(keys, positive), (look, positive)]"
1,4,"[(protector, neutral), (key pad, positive)]"
2,6,"[(mouse, conflict)]"
3,7,"[(printer, positive)]"
4,8,"[(design, positive), (colors, positive)]"


In [ ]:
# --- Task: Join predictions with gold labels on 'id' ---
#
# Using our notebook's actual variable names: result_laptop_df (predictions)
# and true_laptop_df (gold labels).
#
# how="inner" matters here specifically because result_laptop_df is currently
# partial (382/1488, since the classification loop was interrupted) — an
# inner join keeps only ids present in both, i.e. only the reviews we've
# actually classified so far. Once the full loop finishes, this would behave
# the same as a left join since every id would be present in both frames.
eval_df = result_laptop_df.merge(true_laptop_df, on="id", how="inner")

print(f"eval_df rows: {len(eval_df)}")
eval_df.head()


eval_df rows: 382


,id,aspect,aspect_sentiment,true_aspect_sentiment
0,2339,NaN,"[(battery life, positive)]","[(cord, neutral), (battery life, positive)]"
1,1316,NaN,"[(service center, negative), (exchange, negati...","[(service center, negative), (""sales"" team, ne..."
2,2005,NaN,"[(quality, positive), (GUI, positive), (stabil...","[(quality, positive), (GUI, positive), (applic..."
3,2789,NaN,"[(start up, positive), (overheating, neutral)]","[(start up, positive)]"
4,76,NaN,"[(features, positive)]","[(features, positive), (iChat, positive), (Pho..."


# Evaluation Component: LLM-based aspect matching



Rather than hand-writing a fuzzy string-matching rule for aspects (e.g. "battery" vs. "battery life"), use the LLM itself to judge semantic closeness. Build a `PromptTemplate` that takes `true_aspect_sentiments` and `predicted_aspect_sentiments`, and outputs a single number: how many true tuples found a semantically-matching predicted tuple with the same sentiment.

In [ ]:
# --- Task: Build the aspect-matching prompt as a LangChain PromptTemplate ---
#
# Two input variables — true_aspect_sentiments and predicted_aspect_sentiments —
# since this prompt compares two lists rather than classifying a single sentence.
MATCH_PROMPT_TEMPLATE = """You are comparing two lists of (aspect, sentiment) tuples describing the same laptop review: a "true aspect sentiments" list and a "predicted aspect sentiments" list.

For each tuple in the true aspect sentiments list, determine whether there is a matching tuple in the predicted aspect sentiments list. Matching happens in two steps:

1. First, determine whether the first entries of the tuples (the aspects) are describing the same thing. The wording does not need to be identical — only semantically close. For example, "battery" and "battery life" are describing the same thing, while "operating system" and "packaging" are not.
2. If the aspects match, then compare the second entries of the tuples (the sentiments). The sentiments must match exactly — for example, "positive" matches "positive", but "neutral" does not match "negative".

A tuple in the true aspect sentiments list counts as matched if it has at least one matching tuple in the predicted aspect sentiments list (do not count more than one match per true tuple, even if more than one predicted tuple could plausibly match it).

Finally, output the total number of true tuples that found a match, as a single number (e.g. 0, 1, 2, etc.). Output ONLY the number — no explanation or other text.

Example 1:
true aspect sentiments: [('suite of software', 'positive')]
predicted aspect sentiments: [('software', 'positive'), ('suite', 'positive')]
output: 1

Example 2:
true aspect sentiments: [('space', 'positive'), ('keyboard', 'negative')]
predicted aspect sentiments: [('extra space', 'positive'), ('keyboard', 'negative')]
output: 2

Example 3:
true aspect sentiments: [('price premium', 'negative'), ('features', 'positive')]
predicted aspect sentiments: [('price', 'neutral'), ('features', 'positive')]
output: 1

Example 4:
true aspect sentiments: [('web cam', 'neutral'), ("burn cd's", 'neutral')]
predicted aspect sentiments: [('web cam', 'negative'), ('cd burning', 'negative')]
output: 0

Example 5:
true aspect sentiments: [('space', 'positive'), ('keyboard', 'negative')]
predicted aspect sentiments: [('storage', 'positive'), ('screen', 'negative')]
output: 1

Example 6:
true aspect sentiments: [('battery life', 'positive')]
predicted aspect sentiments: [('battery life', 'positive'), ('battery', 'positive')]
output: 1

Now compare the following two lists. Output only the number of matches.

true aspect sentiments: {true_aspect_sentiments}
predicted aspect sentiments: {predicted_aspect_sentiments}
output:"""

match_prompt = PromptTemplate(
    input_variables=["true_aspect_sentiments", "predicted_aspect_sentiments"],
    template=MATCH_PROMPT_TEMPLATE,
)

# --- Quick check: render the prompt using one row from eval_df ---
example_row = eval_df.iloc[0]
rendered_match_prompt = match_prompt.format(
    true_aspect_sentiments=example_row["true_aspect_sentiment"],
    predicted_aspect_sentiments=example_row["aspect_sentiment"],
)
print(rendered_match_prompt)


You are comparing two lists of (aspect, sentiment) tuples describing the same laptop review: a "true aspect sentiments" list and a "predicted aspect sentiments" list.

For each tuple in the true aspect sentiments list, determine whether there is a matching tuple in the predicted aspect sentiments list. Matching happens in two steps:

1. First, determine whether the first entries of the tuples (the aspects) are describing the same thing. The wording does not need to be identical — only semantically close. For example, "battery" and "battery life" are describing the same thing, while "operating system" and "packaging" are not.
2. If the aspects match, then compare the second entries of the tuples (the sentiments). The sentiments must match exactly — for example, "positive" matches "positive", but "neutral" does not match "negative".

A tuple in the true aspect sentiments list counts as matched if it has at least one matching tuple in the predicted aspect sentiments list (do not count mor

In [ ]:
# --- Task: Build eval_chain = eval_prompt | llm | number_parser ---
#
# eval_prompt aliases the match_prompt built above — same object, named to
# match what the chain expects to compose with.
import re

eval_prompt = match_prompt


def number_parser(raw_output: str) -> int | None:
    """
    Extract the first integer found in the LLM's raw output (the match count).
    """
    match = re.search(r"-?\d+", raw_output)
    if match is None:
        print(f"[number_parser] No number found in output: {raw_output!r}")
        return None
    return int(match.group(0))


# A plain function on the right-hand side of "|" is automatically coerced
# into a Runnable by LangChain, so no explicit RunnableLambda wrapper is
# needed here (unlike parse_absc_output, which we wrapped explicitly earlier
# — both forms work identically).
eval_chain = eval_prompt | llm | number_parser

# --- Quick test using the same example row as before ---
example_row = eval_df.iloc[0]
match_count = eval_chain.invoke({
    "true_aspect_sentiments": example_row["true_aspect_sentiment"],
    "predicted_aspect_sentiments": example_row["aspect_sentiment"],
})
print("True:        ", example_row["true_aspect_sentiment"])
print("Predicted:   ", example_row["aspect_sentiment"])
print("Match count: ", match_count)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


True:         [('cord', 'neutral'), ('battery life', 'positive')]
Predicted:    [('battery life', 'positive')]
Match count:  1


In [ ]:
# --- Task: Run eval_chain over every row in eval_df ---
#
# Same loop-and-collect pattern used for the classification loop earlier —
# invoke per row, append to a list, then assign the list as a new column
# once the loop finishes.
from tqdm.auto import tqdm

matches = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating matches"):
    match_count = eval_chain.invoke({
        "true_aspect_sentiments": row["true_aspect_sentiment"],
        "predicted_aspect_sentiments": row["aspect_sentiment"],
    })
    matches.append(match_count)

eval_df["matches"] = matches
eval_df.head()


Evaluating matches: 100%|██████████| 382/382 [2:17:39<00:00, 21.62s/it]


,id,aspect,aspect_sentiment,true_aspect_sentiment,matches
0,2339,NaN,"[(battery life, positive)]","[(cord, neutral), (battery life, positive)]",1
1,1316,NaN,"[(service center, negative), (exchange, negati...","[(service center, negative), (""sales"" team, ne...",0
2,2005,NaN,"[(quality, positive), (GUI, positive), (stabil...","[(quality, positive), (GUI, positive), (applic...",7
3,2789,NaN,"[(start up, positive), (overheating, neutral)]","[(start up, positive)]",1
4,76,NaN,"[(features, positive)]","[(features, positive), (iChat, positive), (Pho...",4


# CPU & Token Monitoring

### Track CPU usage during inference
Since we're running the model locally (no GPU/API offload), it's useful to see how much CPU the model actually consumes per call — this informs whether the chosen model size is practical to run on this machine at dataset scale.

In [ ]:
%pip install -q psutil


In [ ]:
# --- Task: Track CPU usage while the local LLM runs ---
#
# psutil.Process().cpu_percent() reports CPU usage (%) accumulated since the
# last call — a single before/after reading would only capture an average
# over the whole call, so instead we sample repeatedly on a background
# thread while the model is generating, to see how usage varies during a run
# (e.g. a spike at the start when the model is loaded into memory vs. steady
# usage during generation).
import threading
import time

import psutil


def invoke_with_cpu_tracking(chain_or_llm, prompt, sample_interval=0.5):
    """
    Invoke a LangChain LLM/chain while sampling CPU usage in the background.

    Returns (result, cpu_samples) — cpu_samples is a list of dicts so the raw
    readings can be inspected or plotted later, not just the summary stats
    printed here.
    """
    process = psutil.Process()
    cpu_samples = []
    stop_sampling = threading.Event()

    def sample_loop():
        while not stop_sampling.is_set():
            cpu_samples.append({
                # this process only (the Python/Jupyter kernel running the model)
                "process_cpu_percent": process.cpu_percent(interval=None),
                # whole machine, for context (e.g. other apps competing for CPU)
                "system_cpu_percent": psutil.cpu_percent(interval=None),
            })
            time.sleep(sample_interval)

    # The first call to cpu_percent(interval=None) always returns 0.0 or a
    # meaningless value since there's no prior call to measure against — call
    # it once here to establish a baseline before the real sampling starts.
    process.cpu_percent(interval=None)
    psutil.cpu_percent(interval=None)

    sampler_thread = threading.Thread(target=sample_loop, daemon=True)
    sampler_thread.start()

    start_time = time.perf_counter()
    result = chain_or_llm.invoke(prompt)
    elapsed_seconds = time.perf_counter() - start_time

    stop_sampling.set()
    sampler_thread.join()

    if cpu_samples:
        avg_process_cpu = sum(s["process_cpu_percent"] for s in cpu_samples) / len(cpu_samples)
        max_process_cpu = max(s["process_cpu_percent"] for s in cpu_samples)
    else:
        # call finished faster than one sample_interval — nothing was captured
        avg_process_cpu = max_process_cpu = 0.0

    print(f"Elapsed time: {elapsed_seconds:.2f}s")
    print(f"CPU samples collected: {len(cpu_samples)}")
    print(f"Process CPU usage — avg: {avg_process_cpu:.1f}%, max: {max_process_cpu:.1f}%")
    print("(Note: on a multi-core machine, process CPU% can exceed 100% if multiple cores are used.)")

    return result, cpu_samples


# --- Run the smoke-test prompt again, this time with CPU tracking ---
result, cpu_samples = invoke_with_cpu_tracking(
    llm,
    "Classify the sentiment as positive, negative, or neutral: 'The battery life is excellent.'",
)
print("\nModel output:", result)


### Track input/output token counts
Alongside CPU usage, track how many tokens go into the prompt vs. how many the model generates — useful for estimating throughput/cost at dataset scale (e.g. running this across ~2300 rows).

In [ ]:
# --- Task: Track input/output token counts alongside CPU usage ---
#
# We reuse the tokenizer already loaded by the transformers pipeline
# (hf_text_generation_pipeline.tokenizer) rather than loading a second copy,
# since tokenization must match exactly what the model itself uses to give
# accurate counts.

def invoke_with_usage_tracking(chain_or_llm, prompt, tokenizer, sample_interval=0.5):
    """
    Invoke a LangChain LLM/chain while tracking CPU usage and token counts.

    Returns (result, usage) — usage is a dict with elapsed time, CPU stats,
    and token counts, so it can be logged/aggregated across many calls later
    (e.g. when running this over the full dataset).
    """
    process = psutil.Process()
    cpu_samples = []
    stop_sampling = threading.Event()

    def sample_loop():
        while not stop_sampling.is_set():
            cpu_samples.append({
                "process_cpu_percent": process.cpu_percent(interval=None),
                "system_cpu_percent": psutil.cpu_percent(interval=None),
            })
            time.sleep(sample_interval)

    # Prime the CPU% baseline before sampling starts (see previous cell).
    process.cpu_percent(interval=None)
    psutil.cpu_percent(interval=None)

    # Input tokens: how many tokens the model has to process for the prompt
    # itself, counted before generation starts.
    input_token_count = len(tokenizer.encode(prompt))

    sampler_thread = threading.Thread(target=sample_loop, daemon=True)
    sampler_thread.start()

    start_time = time.perf_counter()
    result = chain_or_llm.invoke(prompt)
    elapsed_seconds = time.perf_counter() - start_time

    stop_sampling.set()
    sampler_thread.join()

    # Output tokens: tokenize the generated text with the same tokenizer.
    # This assumes `result` contains only the completion, not prompt+completion —
    # true here since HuggingFacePipeline defaults return_full_text=False for
    # text-generation tasks, so `result` is just what the model generated.
    output_token_count = len(tokenizer.encode(result))

    if cpu_samples:
        avg_process_cpu = sum(s["process_cpu_percent"] for s in cpu_samples) / len(cpu_samples)
        max_process_cpu = max(s["process_cpu_percent"] for s in cpu_samples)
    else:
        avg_process_cpu = max_process_cpu = 0.0

    usage = {
        "elapsed_seconds": elapsed_seconds,
        "cpu_samples": cpu_samples,
        "avg_process_cpu_percent": avg_process_cpu,
        "max_process_cpu_percent": max_process_cpu,
        "input_tokens": input_token_count,
        "output_tokens": output_token_count,
        "total_tokens": input_token_count + output_token_count,
    }

    print(f"Elapsed time: {elapsed_seconds:.2f}s")
    print(f"CPU usage — avg: {avg_process_cpu:.1f}%, max: {max_process_cpu:.1f}%")
    print(f"Input tokens: {input_token_count}")
    print(f"Output tokens: {output_token_count}")
    print(f"Total tokens: {usage['total_tokens']}")

    return result, usage


# --- Run the smoke-test prompt again, this time tracking tokens too ---
result, usage = invoke_with_usage_tracking(
    llm,
    "Classify the sentiment as positive, negative, or neutral: 'The battery life is excellent.'",
    tokenizer=hf_text_generation_pipeline.tokenizer,
)
print("\nModel output:", result)


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Elapsed time: 82.52s
CPU usage — avg: 7.7%, max: 56.5%
Input tokens: 19
Output tokens: 75
Total tokens: 94

Model output: Classify the sentiment as positive, negative, or neutral: 'The battery life is excellent.' The sentiment of the statement "The battery life is excellent" can be classified as positive. This statement expresses satisfaction with the quality and performance of a product's battery, which is typically associated with high-quality electronics and reliable functionality. Therefore, it falls under the category of positive sentiment.
